# **Messages**

Messages are of four types mainly 
* AIMESSAGE : message from LLM
* HUMAN MESSAGE : querry or instruction provided by the Human
* SYSTEM MESSAGE : used to set behavior
* TOOL MESSAGE : Response obtained from a Tool after creating and using it  

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import find_dotenv,load_dotenv
import os 

if os.environ["GOOGLE_API_KEY"]:
    print("API KEY FOUND ")
else:
    print("Provide the API Key First") 

API KEY FOUND 


In [2]:
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [3]:
# How actually message is sent
from langchain_core.messages import HumanMessage

llm.invoke([HumanMessage(content="What is an Ostrich egg ? answer in 10 words only ")])

AIMessage(content='Largest bird egg, known for its thick shell and rich yolk.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ddd9e-fc5b-7320-a711-309d30c339e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 13, 'total_tokens': 30, 'input_token_details': {'cache_read': 0}})

In [4]:
# Lets define a schema 
# where a person sends human message to the llm and llm appends some information i.e combines the llm response of first llm and his own 

from typing import TypedDict,List,Annotated
from operator import add

class graph_schema(TypedDict):
      message_manual=List
      message_auto=Annotated[List,add]
      

In [5]:
# Define Node functions now
from langchain_core.messages import AIMessage

# Create Node Function for post Creation

def create_post(state:graph_schema)->graph_schema:

    # Manuall approach
    message_manual=state['message_manual']
    response_manual=llm.invoke(message_manual).content
    response_manual_ai=AIMessage(content=response_manual)
    state['message_manual']=message_manual+[response_manual_ai]

    # Automated using Reducer But its not that much readable and understanable might confuse us at the end 
    message_auto=state['message_auto']
    response_auto=llm.invoke(message_auto).content
    response_auto_ai=AIMessage(content=response_auto)
    state['response_auto']=[response_auto_ai]

    return state

# Another Node Function implemting curating a post
def curate_post(state:graph_schema)->graph_schema:
    # for manual 
    message_manual=state['message_manual']
    response_manual=llm.invoke(message_manual).content
    response_manual_ai=AIMessage(response_manual)
    state['message_manual']=response_manual_ai

    # for auto using reducer

    message_auto=state["message_auto"]
    response_auto=llm.invoke(message_auto).content
    response_auto_ai=AIMessage(content=response_auto)
    state['response_auto_ai']=response_auto_ai

    return state

